# TP 9 — Du bit au symbole : compter avant de moduler

**Public cible :** étudiant, technicien radio ou NetOps débutant.  
**Durée :** 60 minutes. **Mode :** entièrement hors ligne, synthétique et déterministe.

Question fil rouge : quand une fiche annonce « 16-QAM », le nombre 16 désigne-t-il le nombre de symboles transmis ? **Non.** Il désigne l'ordre de modulation, donc le nombre d'états possibles de la constellation. Le nombre de symboles effectivement transmis dépend des bits, du codage et de la grille radio.

Le TP n'accède à aucun équipement ni réseau. Tous les nombres, constellations et incidents sont produits localement avec la bibliothèque standard Python.

## Contrat d'apprentissage et vocabulaire

À la fin du TP, vous saurez :

1. distinguer **l'ordre** $M$ (nombre d'états possibles) du **compte** $N_{sym}$ (nombre de symboles transmis) ;
2. calculer $k=\log_2(M)$ bits codés par symbole, puis $N_{coded}$, $N_{sym}$ et le **bourrage (padding)** ;
3. relier débit symbole $R_s$ en baud, durée symbole $T_s$, débit binaire codé et débit utile ;
4. lire une constellation Gray BPSK, QPSK ou 16-QAM sans confondre symbole complexe I/Q et échantillon temporel ;
5. compter les symboles de modulation dans une grille OFDM après retrait des pilotes, du contrôle, de la porteuse DC et des bandes de garde ;
6. ne pas confondre $E_b/N_0$, $E_s/N_0$, SNR, EVM et BLER ;
7. expliquer pourquoi augmenter $M$ peut **réduire** le goodput lorsque le canal ne supporte pas le MCS.

Notation : **M = taille de l'alphabet**, **k = bits codés par symbole**, **N_sym = occurrences envoyées**. Dire seulement « nombre de symboles » sans préciser lequel crée l'ambiguïté que ce TP cherche à éliminer.

Repères de vocabulaire : **bits par symbole**, **bit/s**, **bourrage**, **élément de ressource**, **échantillon I/Q**, **Es/N0** et **Eb/N0** désignent des grandeurs différentes ; le TP les relie sans les confondre.

In [ ]:
import base64
import json
import math
from fractions import Fraction

SEED = 1609

class RichHTML:
    def __init__(self, markup):
        self.markup = markup

    def _repr_html_(self):
        return self.markup

    def __repr__(self):
        return "RichHTML(deterministic)"

def html_table(title, columns, rows):
    head = "".join(
        f"<th style='padding:7px 9px;text-align:left;border-bottom:1px solid #475569'>{c}</th>"
        for c in columns
    )
    body = "".join(
        "<tr>" + "".join(
            f"<td style='padding:7px 9px;border-bottom:1px solid #1e293b'>{value}</td>"
            for value in row
        ) + "</tr>"
        for row in rows
    )
    return RichHTML(
        "<div style='font:14px system-ui;color:#e2e8f0;background:#0f172a;"
        "border:1px solid #334155;border-radius:12px;padding:14px;overflow:auto'>"
        f"<h3 style='margin:0 0 10px;color:#7dd3fc'>{title}</h3>"
        f"<table style='border-collapse:collapse;width:100%'><thead><tr>{head}</tr></thead>"
        f"<tbody>{body}</tbody></table></div>"
    )

def json_data_link(filename, payload, label):
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, indent=2).encode("utf-8")
    encoded = base64.b64encode(raw).decode("ascii")
    return (
        f"<a download='{filename}' href='data:application/json;base64,{encoded}' "
        "style='display:inline-block;margin:4px;padding:9px 12px;border-radius:8px;"
        "background:#0ea5e9;color:white;text-decoration:none'>" + label + "</a>"
    )

def ceil_fraction(value):
    return (value.numerator + value.denominator - 1) // value.denominator


## 1. Combien de symboles faut-il transmettre ?

Pour une modulation d'ordre $M$ qui est une puissance de deux :

$$k=\log_2(M),\qquad N_{coded}=\left\lceil\frac{N_{info}}{R_c}\right\rceil,\qquad
N_{sym}=\left\lceil\frac{N_{coded}}{k}\right\rceil$$

- $N_{info}$ : bits d'information avant codage ;
- $R_c$ : rendement du code, par exemple 3/4 ;
- $N_{coded}$ : approximation pédagogique du nombre de bits codés ;
- $N_{sym}$ : nombre de décisions de constellation à mapper ; dans le cas monoporteuse sérialisé ci-dessous, c'est aussi le nombre d'intervalles symbole ;
- **bourrage** : positions binaires ajoutées pour remplir le dernier symbole. Le terme anglais *padding* a été donné une fois dans le contrat d'apprentissage.

**Réserve importante :** un code réel ne produit pas toujours simplement $\lceil N_{info}/R_c\rceil$. Il impose des tailles de bloc, CRC, segmentation, bits de terminaison, poinçonnage ou répétition. La formule isole ici l'idée ; une norme réelle doit fournir la chaîne exacte.

**Périmètre de ce premier calcul :** $N_{sym}/R_s$ décrit une chaîne **monoporteuse sérialisée**, où une décision de constellation occupe chaque intervalle $T_s=1/R_s$. Le baud compte alors ces intervalles par seconde et, sans autre surcharge, $R_{b,coded}=R_s k$. Si seule une fraction temporelle $\eta$ reste disponible après les surcharges autres que le code, le temps mural devient $N_{sym}/(R_s\eta)$ et le débit utile nominal vaut $R_s k R_c\eta$.

En **OFDM**, plusieurs symboles QAM occupent en parallèle des éléments de ressource distincts pendant un même symbole OFDM. On calcule sa durée avec $T_{OFDM}=T_u+T_{CP}$ et le nombre de colonnes temporelles de la grille ; on ne divise pas le total des points QAM par le $R_s$ monoporteuse. La section 3 applique ce second modèle.

In [ ]:
def count_modulation_symbols(n_info, code_rate, order_m, symbol_rate_baud, scheduled_fraction):
    rate = Fraction(code_rate)
    bits_per_symbol = int(math.log2(order_m))
    assert 2 ** bits_per_symbol == order_m, "M doit être une puissance de deux ici"
    coded_bits = ceil_fraction(Fraction(n_info, 1) / rate)
    symbol_count = math.ceil(coded_bits / bits_per_symbol)
    bourrage_bits = symbol_count * bits_per_symbol - coded_bits
    active_modulation_duration_s = symbol_count / symbol_rate_baud
    wall_duration_s = active_modulation_duration_s / scheduled_fraction
    coded_bitrate = symbol_rate_baud * bits_per_symbol
    nominal_useful_bitrate = coded_bitrate * float(rate) * scheduled_fraction
    transaction_goodput = n_info / wall_duration_s
    return {
        "N_info_bits": n_info,
        "Rc": f"{rate.numerator}/{rate.denominator}",
        "M_states": order_m,
        "k_coded_bits_per_symbol": bits_per_symbol,
        "N_coded_bits_model": coded_bits,
        "N_sym_transmitted": symbol_count,
        "bourrage_bits": bourrage_bits,
        "Rs_baud": symbol_rate_baud,
        "Ts_microseconds": round(1e6 / symbol_rate_baud, 3),
        "coded_bitrate_bps": coded_bitrate,
        "active_modulation_duration_ms": round(1000 * active_modulation_duration_s, 6),
        "wall_duration_ms": round(1000 * wall_duration_s, 6),
        "nominal_useful_bitrate_bps": round(nominal_useful_bitrate),
        "transaction_goodput_bps": round(transaction_goodput),
        "scheduled_modulation_fraction": scheduled_fraction,
    }

modulation_report = count_modulation_symbols(
    n_info=1001,
    code_rate=Fraction(3, 4),
    order_m=16,
    symbol_rate_baud=250_000,
    scheduled_fraction=0.82,
)

assert modulation_report["M_states"] == 16
assert modulation_report["k_coded_bits_per_symbol"] == 4
assert modulation_report["N_coded_bits_model"] == 1335
assert modulation_report["N_sym_transmitted"] == 334
assert modulation_report["bourrage_bits"] == 1
assert modulation_report["coded_bitrate_bps"] == 1_000_000
assert modulation_report["wall_duration_ms"] == 1.629268
assert modulation_report["transaction_goodput_bps"] == 614_386

html_table(
    "Exemple : 1 001 bits, Rc=3/4, 16-QAM, Rs=250 kbaud",
    ("Grandeur", "Valeur", "Interprétation"),
    (
        ("M", 16, "16 états possibles, pas 16 occurrences"),
        ("k", 4, "4 bits codés par symbole"),
        ("N_coded", 1335, "ceil(1001 ÷ 3/4), modèle simplifié"),
        ("N_sym", 334, "ceil(1335 ÷ 4) occurrences"),
        ("Bourrage", 1, "une position du dernier symbole ; ensuite : bourrage"),
        ("Rs / Ts monoporteuse", "250 kbaud / 4 µs", "cadence / intervalle actif"),
        ("Temps actif", "1,336 ms", "334 ÷ Rs"),
        ("Durée murale", "1,629 ms", "1,336 ms ÷ 0,82"),
        ("Rb,coded", "1,000 Mbit/s", "Rs × k"),
        ("Débit utile nominal", "615,0 kbit/s", "Rs × k × Rc × 0,82"),
        ("Goodput de la transaction", "614,4 kbit/s", "1 001 bits ÷ 1,629 ms ; écart dû au bourrage"),
    ),
)


## 2. Une constellation est un alphabet, pas une chronologie

Un **symbole complexe** $I+jQ$ est un point choisi dans une constellation. Le mapping Gray place généralement deux étiquettes binaires voisines à une distance de Hamming égale à un, afin qu'une erreur vers un voisin change souvent un seul bit.

- BPSK : $M=2$, donc $k=1$ ;
- QPSK : $M=4$, donc $k=2$ ;
- 16-QAM : $M=16$, donc $k=4$.

Un **symbole de modulation** est une valeur de l'alphabet I/Q ; un **échantillon I/Q** est une observation complexe de la forme d'onde à la fréquence $F_s$. Selon l'endroit de la chaîne, on peut avoir plusieurs échantillons de forme d'onde par symbole ($F_s/R_s>1$), ou une statistique de décision complexe par symbole après filtrage adapté et synchronisation. Cette statistique peut elle-même être stockée comme un échantillon : les deux mots ne sont donc pas des antonymes. Il faut annoncer le point d'observation et le rapport $F_s/R_s$. Suréchantillonner ne crée aucune décision de constellation supplémentaire. En OFDM, l'IFFT produit un bloc d'échantillons temporels, puis la FFT estime en parallèle les symboles complexes des sous-porteuses utiles.

In [ ]:
gray_axis = {"00": -3, "01": -1, "11": 1, "10": 3}
constellations = {
    "BPSK": [("0", -1, 0), ("1", 1, 0)],
    "QPSK": [
        ("00", -1, 1), ("01", 1, 1),
        ("11", 1, -1), ("10", -1, -1),
    ],
    "16-QAM": [
        (i_bits + q_bits, i_value, q_value)
        for q_bits, q_value in gray_axis.items()
        for i_bits, i_value in gray_axis.items()
    ],
}

assert len(constellations["BPSK"]) == 2
assert len(constellations["QPSK"]) == 4
assert len(constellations["16-QAM"]) == 16
assert sum(a != b for a, b in zip("00", "01")) == 1

sample_model = {
    "selected_symbol": {"bits": "0111", "I": -1, "Q": 1},
    "sampling_frequency_over_symbol_rate": 4,
    "time_samples": ["-1.00+1.00j", "-0.92+0.85j", "-0.71+0.60j", "-0.38+0.26j"],
    "warning": "Suite illustrative, non issue d’un filtre : quatre échantillons fictifs de forme d’onde associés à un intervalle ; ce ne sont pas quatre décisions de constellation.",
}

html_table(
    "Ordre, bits et représentation",
    ("Modulation", "M", "k=log2(M)", "Exemple de point"),
    (
        ("BPSK", 2, 1, "bit 1 → I+jQ = 1+0j"),
        ("QPSK", 4, 2, "bits 01 → 1+1j"),
        ("16-QAM Gray", 16, 4, "bits 0111 → -1+1j"),
    ),
)


## 3. OFDM : compter les ressources avant les bits

Dans le modèle de cette grille, **un Resource Element (RE) de données porte un symbole de modulation par couche spatiale**. Ainsi :

$$N_{mod\ symbols}=N_{RE,data}\times N_{layers},\qquad
N_{coded\ bits}=N_{RE,data}\times N_{layers}\times k$$

Tous les RE ne transportent pas des données : certains servent aux pilotes/références, au contrôle, à la porteuse DC ou aux bandes de garde. Dans une colonne temporelle, les symboles QAM des sous-porteuses et couches sont transmis **en parallèle**. La durée d'un symbole OFDM est distincte de ce compte : $T_{OFDM}=T_u+T_{CP}$, puis la durée de l'allocation est le nombre de colonnes multiplié par $T_{OFDM}$. Le préfixe cyclique (**CP**) consomme du temps et réduit le débit, mais ce n'est pas un symbole QAM supplémentaire placé dans la grille fréquentielle.

Attention : une norme réelle définit précisément sa grille, ses signaux de référence, ses réservations, son allocation et son codage. Le petit rectangle ci-dessous sert à apprendre le comptage, pas à reproduire un slot 5G, Wi-Fi ou DVB.

In [ ]:
subcarriers = tuple(range(-5, 6))
ofdm_symbols = tuple(range(6))
pilot_positions = {(1, -3), (1, 3), (3, -3), (3, 3), (5, -3), (5, 3)}
ofdm_cells = []

for time_index in ofdm_symbols:
    for subcarrier in subcarriers:
        if abs(subcarrier) == 5:
            role = "guard"
        elif subcarrier == 0:
            role = "dc"
        elif time_index == 0:
            role = "control"
        elif (time_index, subcarrier) in pilot_positions:
            role = "pilot"
        else:
            role = "data"
        ofdm_cells.append({"symbol_index": time_index, "subcarrier": subcarrier, "role": role})

role_counts = {
    role: sum(cell["role"] == role for cell in ofdm_cells)
    for role in ("data", "pilot", "control", "dc", "guard")
}
layers = 2
order_m = 16
bits_per_symbol = int(math.log2(order_m))
cp_fraction = Fraction(1, 8)
data_re = role_counts["data"]

ofdm_grid = {
    "dimensions": {"subcarriers": len(subcarriers), "ofdm_symbols": len(ofdm_symbols)},
    "role_counts": role_counts,
    "layers": layers,
    "M": order_m,
    "k": bits_per_symbol,
    "data_modulation_symbols": data_re * layers,
    "coded_bits_mapped": data_re * layers * bits_per_symbol,
    "cp_fraction_of_useful_symbol": f"{cp_fraction.numerator}/{cp_fraction.denominator}",
    "cp_time_efficiency": round(float(1 / (1 + cp_fraction)), 3),
    "cells": ofdm_cells,
}

assert sum(role_counts.values()) == len(subcarriers) * len(ofdm_symbols)
assert role_counts == {"data": 34, "pilot": 6, "control": 8, "dc": 6, "guard": 12}
assert ofdm_grid["data_modulation_symbols"] == 68
assert ofdm_grid["coded_bits_mapped"] == 272

html_table(
    "Comptage de la mini-grille OFDM",
    ("Objet", "Compte", "Effet"),
    (
        ("RE totaux", 66, "11 sous-porteuses × 6 symboles OFDM"),
        ("RE de données", 34, "après garde, DC, contrôle et pilotes"),
        ("Couches", 2, "un symbole de modulation par RE de données et couche"),
        ("Symboles QAM", 68, "34 × 2"),
        ("Bits codés mappés", 272, "68 × 4 avec 16-QAM"),
        ("CP", "1/8", "efficacité temporelle 8/9 ; aucun RE ajouté"),
    ),
)


## 4. Énergie, qualité et adaptation MCS

Les indicateurs répondent à des questions différentes :

- **SNR** : rapport puissance signal/bruit sur la mesure considérée ;
- **EVM** : distance RMS entre points reçus et points idéaux ; dans un cas AWGN idéal, $SNR\approx1/EVM^2$, mais les défauts RF rompent cette équivalence simple ;
- **BLER** : fraction des blocs rejetés, généralement après contrôle CRC ;
- **MCS** : couple modulation/codage et paramètres associés que le lien adapte au canal.

Si $E_b$ désigne l'énergie par **bit codé**, $E_s/N_0=E_b/N_0+10\log_{10}(k)$. Si $E_b$ désigne l'énergie par **bit d'information**, et en ignorant les surcharges, $E_s/N_0=E_b/N_0+10\log_{10}(kR_c)$. Il faut donc toujours annoncer la convention. Avec 16-QAM et $R_c=3/4$, les corrections valent respectivement 6,02 dB et 4,77 dB.

Le goodput dépend des bits utiles effectivement livrés. Augmenter $M$ **n'augmente pas intrinsèquement l'EVM** : cela rapproche les points de décision pour une énergie normalisée. À EVM ou SNR donnés, la marge de décision diminue donc et la BLER peut devenir trop forte ; le débit utile peut alors baisser malgré davantage de bits tentés par RE.

In [ ]:
def logistic_bler(snr_db, threshold_db, width_db):
    return 1.0 / (1.0 + math.exp((snr_db - threshold_db) / width_db))

snr_db = 8.0
resource_efficiency = 0.82
mcs_catalog = (
    ("BPSK 1/2", 2, Fraction(1, 2), -1.0, 1.2),
    ("QPSK 1/2", 4, Fraction(1, 2), 2.5, 1.0),
    ("16-QAM 3/4", 16, Fraction(3, 4), 9.0, 1.0),
    ("64-QAM 5/6", 64, Fraction(5, 6), 16.0, 1.1),
)
mcs_results = []
for name, order, rate, threshold, width in mcs_catalog:
    k_value = int(math.log2(order))
    bler = logistic_bler(snr_db, threshold, width)
    attempted = k_value * float(rate) * resource_efficiency
    goodput = attempted * (1 - bler)
    mcs_results.append({
        "name": name,
        "M": order,
        "k": k_value,
        "Rc": f"{rate.numerator}/{rate.denominator}",
        "synthetic_bler": round(bler, 4),
        "attempted_bits_per_resource": round(attempted, 4),
        "goodput_bits_per_resource": round(goodput, 4),
    })

best_mcs = max(mcs_results, key=lambda item: item["goodput_bits_per_resource"])
evm_rms = 10 ** (-snr_db / 20)
energy_conversions = {
    "Es_over_Eb_coded_db_16QAM": round(10 * math.log10(4), 2),
    "Es_over_Eb_information_db_16QAM_Rc_3_4": round(10 * math.log10(4 * 0.75), 2),
    "ideal_awgn_evm_percent_at_8dB": round(100 * evm_rms, 1),
}

assert best_mcs["name"] == "QPSK 1/2"
assert mcs_results[2]["M"] > mcs_results[1]["M"]
assert mcs_results[2]["goodput_bits_per_resource"] < mcs_results[1]["goodput_bits_per_resource"]
assert energy_conversions["Es_over_Eb_coded_db_16QAM"] == 6.02

html_table(
    "Canal synthétique à 8 dB : M plus grand n'est pas toujours meilleur",
    ("MCS", "M", "BLER synthétique", "Bits tentés/ressource", "Goodput/ressource"),
    tuple(
        (
            item["name"], item["M"], f"{100 * item['synthetic_bler']:.1f} %",
            item["attempted_bits_per_resource"], item["goodput_bits_per_resource"],
        )
        for item in mcs_results
    ),
)


In [ ]:
def svg_text(x, y, text, size=14, fill="#e2e8f0", weight=400, anchor="start"):
    safe = str(text).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    return f"<text x='{x}' y='{y}' font-size='{size}' fill='{fill}' font-weight='{weight}' text-anchor='{anchor}'>{safe}</text>"

svg = [
    "<svg xmlns='http://www.w3.org/2000/svg' width='1240' height='980' viewBox='0 0 1240 980' role='img' aria-labelledby='title desc'>",
    "<title id='title'>Du bit au symbole : constellations, calcul et grille OFDM</title>",
    "<desc id='desc'>Tableau pédagogique montrant BPSK, QPSK, 16-QAM, le calcul de 334 symboles, une grille OFDM et le goodput des MCS.</desc>",
    "<rect width='1240' height='980' rx='20' fill='#07111f'/>",
    "<rect x='24' y='22' width='1192' height='70' rx='14' fill='#0f2742' stroke='#1d4ed8'/>",
    svg_text(52, 55, "DU BIT AU SYMBOLE", 24, "#7dd3fc", 700),
    svg_text(52, 78, "Symbole = décision I/Q • échantillons = observations temporelles selon Fs/Rs", 15, "#cbd5e1"),
]

panel_specs = (("BPSK", "BPSK", 42, 120, 310, 260), ("QPSK", "QPSK", 374, 120, 310, 260), ("16-QAM Gray", "16-QAM", 706, 120, 492, 260))
for name, constellation_key, x, y, width, height in panel_specs:
    svg.append(f"<rect x='{x}' y='{y}' width='{width}' height='{height}' rx='14' fill='#0f172a' stroke='#334155'/>")
    svg.append(svg_text(x + 18, y + 28, name, 17, "#f8fafc", 700))
    points = constellations[constellation_key]
    cx, cy = x + width / 2, y + 145
    scale = 47 if name != "16-QAM Gray" else 29
    svg.append(f"<line x1='{x+32}' y1='{cy}' x2='{x+width-32}' y2='{cy}' stroke='#475569'/>")
    svg.append(f"<line x1='{cx}' y1='{y+48}' x2='{cx}' y2='{y+230}' stroke='#475569'/>")
    svg.append(svg_text(x + width - 25, cy - 7, "I", 12, "#94a3b8", 600, "end"))
    svg.append(svg_text(cx + 7, y + 61, "Q", 12, "#94a3b8", 600))
    for label, i_value, q_value in points:
        px, py = cx + i_value * scale, cy - q_value * scale
        svg.append(f"<circle cx='{px}' cy='{py}' r='6' fill='#38bdf8' stroke='#bae6fd'/>")
        label_size = 9 if name == "16-QAM Gray" else 11
        svg.append(svg_text(px + 8, py - 7, label, label_size, "#fef08a", 600))
    svg.append(svg_text(x + 18, y + height - 16, f"M={len(points)}  •  k={int(math.log2(len(points)))} bit(s) codé(s)/symbole", 13, "#a7f3d0", 600))

card_y = 404
cards = (
    (42, "ENTRÉE", "N_info = 1 001 bits", "Rc = 3/4"),
    (278, "CODAGE", "N_coded = 1 335", "modèle simplifié"),
    (514, "16-QAM", "k = log₂(16) = 4", "M = 16 états"),
    (750, "RÉSULTAT", "N_sym = 334", "bourrage = 1 bit"),
    (986, "CADENCE", "Rs = 250 kbaud", "murale = 1,629 ms"),
)
for x, title, line1, line2 in cards:
    color = "#0c4a6e" if title != "RÉSULTAT" else "#14532d"
    svg.append(f"<rect x='{x}' y='{card_y}' width='212' height='94' rx='12' fill='{color}' stroke='#38bdf8'/>")
    svg.append(svg_text(x + 14, card_y + 23, title, 12, "#bae6fd", 700))
    svg.append(svg_text(x + 14, card_y + 52, line1, 16, "#f8fafc", 700))
    svg.append(svg_text(x + 14, card_y + 76, line2, 13, "#dbeafe"))

svg.append(svg_text(42, 538, "MINI-GRILLE OFDM — 1 symbole de modulation par RE de données et par couche", 17, "#f8fafc", 700))
grid_x, grid_y, cell_w, cell_h = 42, 560, 58, 34
role_colors = {"data": "#0284c7", "pilot": "#eab308", "control": "#7c3aed", "dc": "#475569", "guard": "#1e293b"}
role_labels = {"data": "D", "pilot": "P", "control": "C", "dc": "DC", "guard": "G"}
for cell in ofdm_cells:
    col = cell["symbol_index"]
    row = 5 - cell["subcarrier"]
    x, y = grid_x + col * cell_w, grid_y + row * cell_h
    role = cell["role"]
    svg.append(f"<rect x='{x}' y='{y}' width='{cell_w-3}' height='{cell_h-3}' rx='4' fill='{role_colors[role]}' stroke='#64748b'/>")
    svg.append(svg_text(x + (cell_w-3)/2, y + 21, role_labels[role], 11, "#f8fafc", 700, "middle"))
for col in ofdm_symbols:
    svg.append(svg_text(grid_x + col * cell_w + 27, grid_y - 8, f"t{col}", 11, "#94a3b8", 600, "middle"))

legend_x = 420
svg.append(f"<rect x='{legend_x}' y='560' width='778' height='374' rx='14' fill='#0f172a' stroke='#334155'/>")
svg.append(svg_text(legend_x + 20, 589, "COMPTAGE ET QUALITÉ", 17, "#7dd3fc", 700))
legend_items = (("D", "34 RE data"), ("P", "6 pilotes"), ("C", "8 contrôle"), ("DC", "6 DC"), ("G", "12 garde"))
for index, (short, label) in enumerate(legend_items):
    role = ("data", "pilot", "control", "dc", "guard")[index]
    x = legend_x + 20 + index * 145
    svg.append(f"<rect x='{x}' y='606' width='30' height='24' rx='4' fill='{role_colors[role]}'/>")
    svg.append(svg_text(x + 38, 623, label, 11, "#cbd5e1"))

svg.append(svg_text(legend_x + 20, 667, "34 RE data × 2 couches = 68 symboles QAM = 272 bits codés", 18, "#a7f3d0", 700))
svg.append(svg_text(legend_x + 20, 694, "CP = 1/8 : coût temporel, aucun point QAM ajouté à la grille", 14, "#cbd5e1"))
svg.append(svg_text(legend_x + 20, 734, "GOODPUT SYNTHÉTIQUE À SNR = 8 dB", 15, "#f8fafc", 700))
max_goodput = max(item["goodput_bits_per_resource"] for item in mcs_results)
for index, item in enumerate(mcs_results):
    y = 758 + index * 38
    width = 360 * item["goodput_bits_per_resource"] / max_goodput
    fill = "#22c55e" if item["name"] == best_mcs["name"] else "#0ea5e9"
    svg.append(svg_text(legend_x + 20, y + 16, item["name"], 12, "#cbd5e1"))
    svg.append(f"<rect x='{legend_x+150}' y='{y}' width='360' height='23' rx='5' fill='#1e293b'/>")
    svg.append(f"<rect x='{legend_x+150}' y='{y}' width='{width:.1f}' height='23' rx='5' fill='{fill}'/>")
    svg.append(svg_text(legend_x + 525, y + 16, f"{item['goodput_bits_per_resource']:.3f} bit/ress.", 12, "#f8fafc"))
svg.append(svg_text(legend_x + 20, 920 - 15, "Conclusion : à 8 dB, QPSK 1/2 livre plus que 16-QAM 3/4 ou 64-QAM 5/6.", 13, "#fef08a", 600))
svg.append("</svg>")

modulation_svg = "".join(svg)
assert modulation_svg.startswith("<svg")
assert "N_sym = 334" in modulation_svg
assert "34 RE data × 2 couches" in modulation_svg
RichHTML(modulation_svg)


## 5. Incidents et contre-exercices

Avant d'afficher le rapport, diagnostiquez chaque phrase :

1. « 16-QAM transmet 16 symboles. » — confond $M$ et $N_{sym}$.
2. « J'ai 34 RE de données, donc 34 symboles QAM. » — faux ici avec deux couches : il y en a 68.
3. « Quatre échantillons par symbole multiplient par quatre les bits transportés. » — faux : ce sont quatre observations temporelles, sauf si l'on parle explicitement d'une statistique de décision par symbole ; le rapport $F_s/R_s$ ne crée pas quatre décisions.
4. « Passons de QPSK à 64-QAM, le débit utile augmentera forcément. » — faux : $M$ ne crée pas l'EVM, mais les points plus rapprochés tolèrent moins d'erreur à EVM/SNR donnés ; la BLER peut annuler le gain.
5. « Le CP ajoute un symbole QAM. » — faux : il répète une partie du signal temporel et coûte du temps.

**Contre-expérience :** conservez la même grille et le même SNR synthétique, puis comparez bits tentés et goodput. Une configuration peut tenter davantage de bits tout en en livrant moins. C'est précisément la raison d'être de l'adaptation MCS.

In [ ]:
incident_report = {
    "title": "Un lien 16-QAM livre moins que prévu",
    "observed": {
        "snr_db": snr_db,
        "selected_mcs": "16-QAM 3/4",
        "synthetic_bler": mcs_results[2]["synthetic_bler"],
        "goodput_bits_per_resource": mcs_results[2]["goodput_bits_per_resource"],
    },
    "false_hypothesis": "M=16 signifie 16 symboles transmis et garantit un débit supérieur.",
    "falsification": {
        "M_vs_N_sym": "M=16 états ; l'exemple exige N_sym=334 occurrences.",
        "mcs_comparison": "À 8 dB dans ce modèle, QPSK 1/2 a le meilleur goodput.",
        "ofdm_accounting": "Seuls les RE de données, multipliés par les couches, portent les symboles QAM comptés.",
    },
    "decision": "Revenir au MCS QPSK 1/2 tant que la BLER 16-QAM reste excessive.",
    "evidence_scope": "Simulation pédagogique synthétique, pas une mesure RF ni un seuil de production.",
}

full_report = {
    "seed": SEED,
    "bit_to_symbol": modulation_report,
    "constellation_sizes": {name: len(points) for name, points in constellations.items()},
    "sample_model": sample_model,
    "ofdm": {key: value for key, value in ofdm_grid.items() if key != "cells"},
    "energy_conversions": energy_conversions,
    "mcs_scenario": {"snr_db": snr_db, "results": mcs_results, "best": best_mcs["name"]},
    "incident": incident_report,
}

assert full_report["bit_to_symbol"]["N_sym_transmitted"] == 334
assert full_report["ofdm"]["data_modulation_symbols"] == 68
assert full_report["mcs_scenario"]["best"] == "QPSK 1/2"

export_panel = (
    "<div style='font:14px system-ui;color:#e2e8f0;background:#0f172a;"
    "border:1px solid #334155;border-radius:12px;padding:14px'>"
    "<h3 style='margin:0 0 8px;color:#7dd3fc'>Rapports transportables</h3>"
    "<p>Exports JSON embarqués dans la page : aucun fichier ni service externe n'est requis.</p>"
    + json_data_link("tp_modulation_rapport.json", full_report, "Télécharger le rapport complet")
    + json_data_link("tp_modulation_incident.json", incident_report, "Télécharger l'incident")
    + "</div>"
)
RichHTML(export_panel)


## 6. Transfert, vérification et limites

### Défi de transfert

Pour 1 500 bits d'information, $R_c=2/3$ et QPSK : calculez $N_{coded}$, $N_{sym}$ et le bourrage avec le même modèle monoporteuse sérialisé. Puis, sur une allocation de 120 RE de données et quatre couches, comptez les symboles QPSK et les bits codés mappés. Enfin, expliquez en une phrase pourquoi doubler $F_s/R_s$ ne double aucun de ces deux comptes, puis indiquez pourquoi la durée de la grille OFDM dépend de ses colonnes et de $T_{OFDM}$ plutôt que des 480 points QPSK parallèles.

Résultats attendus pour vous auto-corriger : $N_{coded}=2250$, $N_{sym}=1125$, bourrage nul ; 480 symboles QPSK et 960 bits codés mappés.

### Checklist avant d'annoncer un débit

- Ai-je distingué $M$, $k$, $N_{sym}$ et le nombre d'échantillons ?
- Ai-je précisé si $E_b$ désigne un bit codé ou un bit d'information ?
- Ai-je retiré pilotes, contrôle, DC et gardes, puis multiplié par les couches ?
- Ai-je compté CP, trames, CRC, codage et retransmissions dans l'efficacité ?
- Ai-je mesuré le goodput et la BLER au lieu de conclure avec le seul ordre de modulation ?

### Limites

Ce TP est un modèle synthétique : constellations non normalisées en énergie, mapping simplifié, grille OFDM générique, BLER logistique inventée et relation EVM/SNR idéale. Il n'implémente ni mise en forme d'impulsion, synchronisation, FFT, canal radio, code correcteur réel, HARQ, allocation normalisée, ni front-end RF. Les seuils MCS ne doivent jamais guider un réseau réel. Le notebook ne constitue ni preuve de production ni mesure terrain ; il sert à construire un raisonnement vérifiable et à préparer une simulation ou une mesure autorisée.